# Store Sales
## Feature Engineering
* **Goal:** Prepare and transform the dataset for property price prediction

This notebook prepares the cleaned dataset for machine learning models. 

In [43]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [44]:
# Define path

data_path = Path('../data')

# Find CSV
files = list(data_path.glob('*.csv'))

# Load files into dictonary
dataframes = {}

for file in files:
    name = file.stem
    dataframes[name] = pd.read_csv(file)

In [45]:
# Convert date cols to datatime format

for name, df in dataframes.items():
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

In [46]:
# Display the shape of each dataset

for name, df in dataframes.items():
    print(f'{name}: {df.shape}')

test: (28512, 5)
train: (3000888, 6)
transactions: (83488, 3)
oil: (1218, 2)
holidays_events: (350, 6)
sample_submission: (28512, 2)
stores: (54, 5)


In [47]:
# Create a copy of the train data 
train = dataframes['train'].copy()

train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [48]:
# Display shape, cols and missing values
print('Shape:', train.shape)
print(train.columns.tolist())
print(train.isna().sum())


Shape: (3000888, 6)
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64


In [49]:
# Add Store information to train dataset
train = train.merge(
    dataframes['stores'], 
    on='store_nbr', 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}')
print(train.head())

Shape: (3000888, 10)
   id       date  store_nbr      family  sales  onpromotion   city      state  \
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0  Quito  Pichincha   
1   1 2013-01-01          1   BABY CARE    0.0            0  Quito  Pichincha   
2   2 2013-01-01          1      BEAUTY    0.0            0  Quito  Pichincha   
3   3 2013-01-01          1   BEVERAGES    0.0            0  Quito  Pichincha   
4   4 2013-01-01          1       BOOKS    0.0            0  Quito  Pichincha   

  type  cluster  
0    D       13  
1    D       13  
2    D       13  
3    D       13  
4    D       13  


In [50]:
# Add transaction data
train = train.merge(
    dataframes['transactions'], 
    on=['date', 'store_nbr'], 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}')

# Check missing values for transaction
print(f'Missing transactions: {train['transactions'].isna().sum()}')

Shape: (3000888, 11)
Missing transactions: 245784


In [51]:
# Add oil price datas
train = train.merge(
    dataframes['oil'], 
    on='date', 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}]')
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Shape: (3000888, 12)]
Missing oil prices: 928422


In [52]:
# Extract time-based features from date cols
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train['day_of_week'] = train['date'].dt.dayofweek

# Display new features
train[['date', 'year', 'month', 'day', 'day_of_week']].head()

,date,year,month,day,day_of_week
0,2013-01-01,2013,1,1,1
1,2013-01-01,2013,1,1,1
2,2013-01-01,2013,1,1,1
3,2013-01-01,2013,1,1,1
4,2013-01-01,2013,1,1,1


In [55]:
# Identify weekends
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)

# Extract quarter
train['quarter'] = train['date'].dt.quarter

# Display new featrues
train[['date', 'day_of_week', 'is_weekend', 'quarter']].head()

,date,day_of_week,is_weekend,quarter
0,2013-01-01,1,0,1
1,2013-01-01,1,0,1
2,2013-01-01,1,0,1
3,2013-01-01,1,0,1
4,2013-01-01,1,0,1
